# The 4-month cycle by commercial region (Nico)

**Starting point:** `features_ludovico` found a 4-month cycle in the repeat rate (peaks in March, July, November with a low repeat rate; high repeat rate two months before each peak) and added the **month of purchase** to every row. V03 uses it.

**New finding:** the cycle is not shared by every customer. It lives almost only in **commercial region RG17** (`clientes.csv`), which holds 63% of the train rows and 53% of the test rows. The month feature of V03 is the same for every customer, so it averages a strong effect (RG17) with no effect (the other regions).

**Three candidate features, tested one at a time** against V03:

| Feature | What it is | Hypothesis |
|---|---|---|
| `rg17` | 1 if the customer's commercial region is RG17 | the region alone shifts the repeat rate |
| `month × region` | the month of purchase split by region: `3_RG17`, `3_other`, … (replaces the month) | lets a linear model give the cycle a strong weight in RG17 and none elsewhere |
| `past peak share` | share of the customer's **earlier** purchases that fell in a peak month | "peak buyers" exist in every region; their own history says so |

**Leakage:** the region is a static attribute of the customer; the peak share uses only purchases made **before** the current one (dates only, no target).

**Instruments and rule** (as in `preprocessing_hybrid_nico`): the V03 model (logistic regression + L1) and the tuned gradient boosting of `tuning_ludovico`; ROC AUC and honest F1 on the official folds, ROC AUC on the 8 half-year folds. A feature is kept only if it is better than V03 beyond 2 SEM on the official folds, not worse on the 8 folds, and its honest F1 is not lower.

# Setup
`%run` executes `preprocessing_ludovico.ipynb` (about 2 minutes).

In [1]:
%%capture
%run ./preprocessing_ludovico.ipynb

In [2]:
from sklearn.feature_selection import SelectFromModel

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

ALWAYS_DROP = ["customer_id", "purchase_date", "random_noise"]
PEAK_MONTHS = [3, 7, 11]
X_full = train.drop(columns=[TARGET])
X_test_full = test.copy()
early = np.concatenate([va for _, va in official_folds[:-1]])
last = official_folds[-1][1]

clientes = pd.read_csv(DATA_RAW / "clientes.csv").set_index("customer_id")
REGION = clientes["commercial_region"].str.extract(r"(RG\d+)")[0].fillna("missing")

# 1. The pattern
Share of purchases in the peak months and repeat rate by phase of the cycle, for the main regions (train only).

In [3]:
phase_of = {m: "peak (3, 7, 11)" for m in PEAK_MONTHS} | {m: "pre-peak (1, 5, 9)" for m in [1, 5, 9]}
look = X_full.assign(region=X_full["customer_id"].map(REGION), target=y,
                     phase=X_full["purchase_date"].dt.month.map(phase_of).fillna("other months"))
main = look["region"].value_counts().loc[lambda s: s >= 150].index
look["region"] = look["region"].where(look["region"].isin(main), "other regions")

share = pd.crosstab(look["region"], look["phase"], normalize="index")
repeat = look.pivot_table(index="region", columns="phase", values="target", aggfunc="mean")
pattern = pd.concat({"share of purchases": share, "repeat rate": repeat}, axis=1)
pattern[("", "rows")] = look["region"].value_counts()
pattern[("", "customers")] = look.groupby("region")["customer_id"].nunique()
pattern.sort_values(("", "rows"), ascending=False).style.format("{:.2f}").format("{:.0f}", subset=[("", "rows"), ("", "customers")])

**Reading**
- **RG17 buys in the peak months twice as often as the others:** 50% of its purchases fall in March, July or November, against 22–28% in every other region (25% would be a uniform spread over the year).
- **The repeat cycle exists only in RG17:** after a peak-month purchase its customers come back within 90 days 42% of the time, after a pre-peak purchase (January, May, September) 76%. In the other regions the three phases differ by a few points only, and not always in the same direction (RG31 is the only other one with a visible dip).
- RG17 is the largest region (301 customers, 63% of train rows, 53% of test rows), so the cycle found on all rows in `features_ludovico` is mostly its cycle, diluted by the others.

# 2. The three features

In [4]:
# Past peak share: computed once on train and test together, from the dates only
rows = (pd.concat([X_full, X_test_full])[["customer_id", "purchase_date"]]
        .sort_values(["customer_id", "purchase_date"], kind="stable"))
rows["peak"] = rows["purchase_date"].dt.month.isin(PEAK_MONTHS).astype(float)
PAST_PEAK_SHARE = rows.groupby("customer_id")["peak"].transform(lambda s: s.shift(1).expanding().mean())


def base_features(df):
    """V03: the month of purchase (features_ludovico), identifiers dropped."""
    return df.assign(purchase_month=df["purchase_date"].dt.month.astype(str))


def with_rg17(df):
    return base_features(df).assign(rg17=(df["customer_id"].map(REGION) == "RG17").astype(int))


def with_month_x_region(df):
    group = np.where(df["customer_id"].map(REGION) == "RG17", "RG17", "other")
    return base_features(df).assign(purchase_month=df["purchase_date"].dt.month.astype(str) + "_" + group)


def with_past_peak_share(df):
    return base_features(df).assign(past_peak_share=PAST_PEAK_SHARE.reindex(df.index))


FEATURES = {
    "V03 (month)": base_features,
    "+ rg17": with_rg17,
    "month × region (replaces month)": with_month_x_region,
    "+ past peak share": with_past_peak_share,
}


def drop_ids(make):
    return lambda df: make(df).drop(columns=[c for c in ALWAYS_DROP if c in df.columns])


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


l1_selector = SelectFromModel(
    make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1, solver="saga", C=0.05, max_iter=5000)),
    importance_getter="named_steps.logisticregression.coef_",
)
TUNED_BOOSTING = dict(min_samples_leaf=200, max_leaf_nodes=31, max_iter=100, max_features=0.5,
                      learning_rate=0.05, l2_regularization=10.0)  # tuning_ludovico, logged with V05


def v03_model(make):
    steps = linear_recipe(LogisticRegression(max_iter=2000)).set_params(
        prep__num__transform=FunctionTransformer(signed_log1p, feature_names_out="one-to-one")).steps
    return Pipeline([("features", FunctionTransformer(drop_ids(make))), *steps[:-1],
                     ("select", clone(l1_selector)), steps[-1]])


def boosting_model(make):
    steps = tree_recipe(HistGradientBoostingClassifier(random_state=RANDOM_STATE, **TUNED_BOOSTING)).steps
    return Pipeline([("features", FunctionTransformer(drop_ids(make))), *steps])


for name, make in FEATURES.items():
    added = make(X_full.head(50)).columns.difference(X_full.columns)
    make(X_test_full.head(50))
    print(f"{name:35s} -> new or changed columns: {', '.join(added)}")
print("past peak share: NaN for", PAST_PEAK_SHARE.isna().sum(), "first purchases;",
      "test rows with a value:", PAST_PEAK_SHARE.reindex(X_test_full.index).notna().mean().round(3))

V03 (month)                         -> new or changed columns: purchase_month
+ rg17                              -> new or changed columns: purchase_month, rg17
month × region (replaces month)     -> new or changed columns: purchase_month
+ past peak share                   -> new or changed columns: past_peak_share, purchase_month
past peak share: NaN for 629 first purchases; test rows with a value: 0.978


**Check:** the reference must give back V03 exactly (ROC AUC 0.7958 on the official folds).

In [5]:
v03_scores, _ = evaluate(v03_model(base_features), official_folds, X_full, y)
assert abs(v03_scores["ROC AUC"].mean() - 0.7958) < 5e-4, v03_scores["ROC AUC"].mean()
print("reference reproduces V03:", round(v03_scores["ROC AUC"].mean(), 4))

reference reproduces V03: 0.7958


# 3. One feature at a time

In [6]:
def paired_vs(a, b):
    d = np.asarray(a) - np.asarray(b)
    return d.mean(), d.std(ddof=1) / np.sqrt(len(d)), f"{(d > 0).sum()}/{len(d)}"


def board(model, features):
    """Official folds (ROC AUC, honest F1) and 8 half-year folds (ROC AUC), each paired with the first entry."""
    official, half_year, rows = {}, {}, {}
    for name, make in features.items():
        scores, oof = evaluate(model(make), official_folds, X_full, y)
        official[name] = scores["ROC AUC"].to_numpy()
        half_year[name] = evaluate(model(make), prep_folds, X_full, y)[0]["ROC AUC"].to_numpy()
        t, _ = best_threshold(y.iloc[early], oof.iloc[early])
        rows[name] = {"ROC AUC": official[name].mean(), "fold 1 / 2 / 3": " / ".join(f"{v:.3f}" for v in official[name]),
                      "honest F1": f1_score(y.iloc[last], oof.iloc[last] >= t), "threshold": t}
    reference = next(iter(features))
    for name in rows:
        diff, sem, won = paired_vs(official[name], official[reference])
        diff8, sem8, won8 = paired_vs(half_year[name], half_year[reference])
        rows[name] |= {"diff vs V03": diff, "SEM": sem, "folds won": won,
                       "8 folds: diff": diff8, "8 folds: SEM": sem8, "8 folds won": won8}
    return pd.DataFrame(rows).T


FORMAT = {"ROC AUC": "{:.4f}", "honest F1": "{:.3f}", "threshold": "{:.2f}", "diff vs V03": "{:+.4f}", "SEM": "{:.4f}",
          "8 folds: diff": "{:+.4f}", "8 folds: SEM": "{:.4f}"}
linear_board = board(v03_model, FEATURES)
linear_board.style.format(FORMAT)

,ROC AUC,fold 1 / 2 / 3,honest F1,threshold,diff vs V03,SEM,folds won,8 folds: diff,8 folds: SEM,8 folds won
V03 (month),0.7958,0.779 / 0.807 / 0.801,0.802,0.43,+0.0000,0.0000,0/3,+0.0000,0.0000,0/8
+ rg17,0.7944,0.788 / 0.794 / 0.801,0.800,0.39,-0.0013,0.0065,2/3,-0.0002,0.0020,3/8
month × region (replaces month),0.8019,0.796 / 0.800 / 0.810,0.798,0.43,+0.0062,0.0072,2/3,+0.0055,0.0024,7/8
+ past peak share,0.7951,0.780 / 0.803 / 0.802,0.804,0.38,-0.0007,0.0017,2/3,-0.0025,0.0025,3/8


**Reading (V03 model: logistic regression + L1)**
- **`month × region` is the only feature with a consistent gain in ranking:** +0.0062 ROC AUC on the official folds and **+0.0055 ± 0.0024 on the 8 half-year folds, better on 7 of 8 (2.3 SEM)**. On the official folds it is noisier: +0.017 on 2022, −0.007 on 2023, +0.009 on 2024–25, so the 2-SEM bar is not reached there (±0.0072).
- **Its honest F1 is slightly lower** (0.798 against 0.802) at the same threshold (0.43): on fold 3 the ranking improves (ROC AUC 0.801 → 0.810) but the cut at 0.43 lands on a few more errors. A difference of 0.004 on one fold of 835 rows is within the noise of F1.
- **`rg17` alone adds nothing** (−0.0013): the region shifts the level of the repeat rate only a little; what it changes is the *effect of the month*, which a flag cannot express in a linear model.
- **`past peak share` adds nothing** (−0.0007 official, −0.0025 on 8 folds): being a peak buyer is already reflected in the region and in the purchase frequency.

In [7]:
boosting_board = board(boosting_model, FEATURES)
boosting_board.style.format(FORMAT)

,ROC AUC,fold 1 / 2 / 3,honest F1,threshold,diff vs V03,SEM,folds won,8 folds: diff,8 folds: SEM,8 folds won
V03 (month),0.8008,0.786 / 0.805 / 0.812,0.799,0.38,+0.0000,0.0000,0/3,+0.0000,0.0000,0/8
+ rg17,0.8026,0.789 / 0.805 / 0.814,0.801,0.38,+0.0018,0.0010,3/3,+0.0004,0.0020,3/8
month × region (replaces month),0.8012,0.797 / 0.798 / 0.808,0.788,0.44,+0.0004,0.0054,1/3,-0.0000,0.0023,4/8
+ past peak share,0.8014,0.792 / 0.800 / 0.812,0.785,0.46,+0.0006,0.0031,1/3,+0.0000,0.0022,3/8


**Reading (tuned gradient boosting)**
- The tuned boosting can already combine month and other columns in its trees, so the interaction gives it nothing (+0.0004).
- **`rg17` is the only small positive for it:** +0.0018 ± 0.0010 on the official folds (3/3), but +0.0004 on the 8 folds: inside the noise. Its honest F1 changes little (0.799 → 0.801).
- `past peak share` lowers its honest F1 (0.785) through the threshold, with an unchanged ranking.

# 4. Decision
The rule of the introduction, applied by code to the V03 model.

In [8]:
passed = [name for name, row in linear_board.iterrows() if name != "V03 (month)"
          and row["diff vs V03"] > 2 * row["SEM"] and row["8 folds: diff"] >= 0
          and row["honest F1"] >= linear_board.loc["V03 (month)", "honest F1"]]
print("features that pass the rule:", passed or "none")

features that pass the rule: none


# 5. Summary

| Feature (added to V03) | V03 model: ROC AUC vs V03 (official / 8 folds) | Honest F1 | Tuned boosting: ROC AUC vs reference (official / 8 folds) |
|---|---|---|---|
| V03 (month) | – | 0.802 | – |
| `rg17` | −0.0013 / −0.0002 | 0.800 | +0.0018 / +0.0004 |
| **`month × region`** | **+0.0062 / +0.0055 (7/8 folds, 2.3 SEM)** | 0.798 | +0.0004 / −0.0000 |
| `past peak share` | −0.0007 / −0.0025 | 0.804 | +0.0006 / +0.0000 |

- **No feature passes the rule declared at the start.** The closest is `month × region` for the linear model: its gain in ROC AUC is the most consistent seen in any feature experiment (7 of 8 half-year folds), but on the 3 official folds it is too noisy to clear 2 SEM, and its honest F1 is 0.004 lower.
- **The finding itself is solid and useful for the report:** the 4-month cycle is a behaviour of RG17 customers (50% of their purchases in peak months, repeat rate 42% after a peak purchase vs 76% before it), not of the whole customer base.
- **Decision:** V03 stays the reference. `month × region` is the best candidate if the team wants one more exploratory submission: from its honest F1, the expected Kaggle score is close to V03's (about 0.80).

### Lines for `docs/decisions.md`

| Decision | Alternatives considered | Evidence |
|---|---|---|
| Keep the month as in V03 (not split by region) | month × commercial region (RG17 / other); RG17 flag; customer's past share of peak-month purchases | `features_region_nico` §3: month × region +0.0062 ROC AUC official (2/3, under 2 SEM), +0.0055 on 8 folds (7/8), honest F1 0.798 vs 0.802; the other two within ±0.003 |
| The 4-month cycle is a feature of region RG17 | – | §1: 50% of RG17 purchases in peak months (others 22–28%); RG17 repeat rate 0.42 after a peak purchase vs 0.76 before one; other regions flat |